In [62]:
import sys
!{sys.executable} -m pip install pandas



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import requests


In [6]:
from pprint import pprint 


In [7]:
import pandas as pd

In [12]:
response = requests.get("http://rest.uniprot.org/uniprotkb/p0dmv8") 
print(type(response))

<class 'requests.models.Response'>


In [13]:
record = response.json()
pprint(record)

{'annotationScore': 5.0,
 'comments': [{'commentType': 'FUNCTION',
               'texts': [{'evidences': [{'evidenceCode': 'ECO:0000250',
                                         'id': 'P0DMW0',
                                         'source': 'UniProtKB'},
                                        {'evidenceCode': 'ECO:0000269',
                                         'id': '22528486',
                                         'source': 'PubMed'},
                                        {'evidenceCode': 'ECO:0000269',
                                         'id': '23973223',
                                         'source': 'PubMed'},
                                        {'evidenceCode': 'ECO:0000269',
                                         'id': '24318877',
                                         'source': 'PubMed'},
                                        {'evidenceCode': 'ECO:0000269',
                                         'id': '24613385',
                             

In [20]:
params = {
    'fields': ["accession", "gene_names", "id", "length", "protein_name",]
} 

response = requests.get("http://rest.uniprot.org/uniprotkb/p0dmv8", params = params) 
pprint(response.json())

{'entryType': 'UniProtKB reviewed (Swiss-Prot)',
 'extraAttributes': {'uniParcId': 'UPI0000000C40'},
 'genes': [{'geneName': {'value': 'HSPA1A'},
            'synonyms': [{'evidences': [{'evidenceCode': 'ECO:0000303',
                                         'id': '24318877',
                                         'source': 'PubMed'}],
                          'value': 'HSP72'},
                         {'value': 'HSPA1'},
                         {'value': 'HSX70'}]}],
 'primaryAccession': 'P0DMV8',
 'proteinDescription': {'alternativeNames': [{'fullName': {'value': 'Heat '
                                                                    'shock 70 '
                                                                    'kDa '
                                                                    'protein '
                                                                    '1'},
                                              'shortNames': [{'evidences': [{'evidenceCode': 'ECO:0000303',

In [17]:

accession = "P0DMV8"
url = f"https://www.uniprot.org/uniprot/{accession}.fasta"
response = requests.get(url)
print(response.text)

>sp|P0DMV8|HS71A_HUMAN Heat shock 70 kDa protein 1A OS=Homo sapiens OX=9606 GN=HSPA1A PE=1 SV=1
MAKAAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYVAFTDTERLIGDAAKNQVA
LNPQNTVFDAKRLIGRKFGDPVVQSDMKHWPFQVINDGDKPKVQVSYKGETKAFYPEEIS
SMVLTKMKEIAEAYLGYPVTNAVITVPAYFNDSQRQATKDAGVIAGLNVLRIINEPTAAA
IAYGLDRTGKGERNVLIFDLGGGTFDVSILTIDDGIFEVKATAGDTHLGGEDFDNRLVNH
FVEEFKRKHKKDISQNKRAVRRLRTACERAKRTLSSSTQASLEIDSLFEGIDFYTSITRA
RFEELCSDLFRSTLEPVEKALRDAKLDKAQIHDLVLVGGSTRIPKVQKLLQDFFNGRDLN
KSINPDEAVAYGAAVQAAILMGDKSENVQDLLLLDVAPLSLGLETAGGVMTALIKRNSTI
PTKQTQIFTTYSDNQPGVLIQVYEGERAMTKDNNLLGRFELSGIPPAPRGVPQIEVTFDI
DANGILNVTATDKSTGKANKITITNDKGRLSKEEIERMVQEAEKYKAEDEVQRERVSAKN
ALESYAFNMKSAVEDEGLKGKISEADKKKVLDKCQEVISWLDANTLAEKDEFEHKRKELE
QVCNPIISGLYQGAGGPGPGGFGAQGPKGGSGSGPTIEEVD



In [8]:
query = (
    'family:"Heat shock protein 70 family" AND reviewed:true AND '
    '(organism_id:9606 OR organism_id:10090 OR organism_id:3702 OR organism_id:83333)') 

In [9]:
params = {
    'query': query,
}
response = requests.get("http://rest.uniprot.org/uniprotkb/search", params = params)

In [10]:
results = response.json()['results']
print(len(results))

25


In [13]:
def print_summary_line(record):
    """ 
    Print a summary line for a UniProt record.
    
    Args:
        record (dict): A UniProt record as a dictionary.

    Returns:
        None
    """

    up_id = record.get('uniProtkbId', 'N/A') 

    try:
        protein_name = record['proteinDescription']['recommendedName']['fullName']['value'] 
    except KeyError: 
        protein_name="N/A" 

    try: 
        gene_name = record['genes'][0]['geneName']['value'] 
    except KeyError: 
        gene_name="N/A" 

    try: 
        length = record['sequence']['length'] 
    except KeyError: 
        length="N/A"
          
    try:
         description = record['proteinDescription']['recommendedName']['fullName']['value']
    except KeyError:
         description = "N/A"

    try:
        organism = record['organism']['scientificName']
    except KeyError:
        organism = "N/A" 

    try:
        function = "N/A"
        for comment in record['comments']:
            if comment.get('commentType') == 'FUNCTION':
                function = comment['texts'][0]['value']
                break
    except (KeyError, IndexError, TypeError):
        function = "N/A"    

    try:
        location = "N/A"
        for comment in record['comments']:
            if comment.get('commentType') == 'SUBCELLULAR LOCATION':
                location = comment['subcellularLocations'][0]['location']['value']
                break
    except (KeyError, IndexError, TypeError):
        location = "N/A"


    try:
        taxon_id = record['organism']['taxonId']
    except KeyError:
        taxon_id = "N/A"

    print(f"{up_id}, {protein_name}, {gene_name}, {length}, {description}, {organism}, {function}, {location}, ({taxon_id})")

In [14]:
for record in results:
    print_summary_line(record)

HSP7E_HUMAN, Heat shock 70 kDa protein 14, HSPA14, 509, Heat shock 70 kDa protein 14, Homo sapiens, Component of the ribosome-associated complex (RAC), a complex involved in folding or maintaining nascent polypeptides in a folding-competent state. In the RAC complex, binds to the nascent polypeptide chain, while DNAJC2 stimulates its ATPase activity, Cytoplasm, cytosol, (9606)
HS71L_HUMAN, Heat shock 70 kDa protein 1-like, HSPA1L, 641, Heat shock 70 kDa protein 1-like, Homo sapiens, Molecular chaperone implicated in a wide variety of cellular processes, including protection of the proteome from stress, folding and transport of newly synthesized polypeptides, activation of proteolysis of misfolded proteins and the formation and dissociation of protein complexes. Plays a pivotal role in the protein quality control system, ensuring the correct folding of proteins, the re-folding of misfolded proteins and controlling the targeting of proteins for subsequent degradation. This is achieved th

In [15]:
def get_record_dict(record):
    """ 
    Extract relevant fields from a UniProt record and return them as a dictionary.
    
    Args:
        record (dict): A UniProt record as a dictionary.

    Returns:
        dict: A dictionary containing the extracted fields.
    """

    # extract relevant fields from the record, substituting "N/A" if a field is missing
    up_id = record.get('primaryAccession', 'N/A')
    entry_name = record.get('uniProtkbId', 'N/A')

    try:
        protein_name = record['proteinDescription']['recommendedName']['fullName']['value'] 
    except KeyError: 
        protein_name="N/A" 

    try: 
        gene_name = record['genes'][0]['geneName']['value'] 
    except KeyError: 
        gene_name="N/A"   

    try: 
        length = record['sequence']['length'] 
    except KeyError: 
        length="N/A"  

    try:
         description = record['proteinDescription']['recommendedName']['fullName']['value']
    except KeyError:
         description = "N/A"

    try:
        organism = record['organism']['scientificName']
    except KeyError:
        organism = "N/A" 

    try:
        function = "N/A"
        for comment in record['comments']:
            if comment.get('commentType') == 'FUNCTION':
                function = comment['texts'][0]['value']
                break
    except (KeyError, IndexError, TypeError):
        function = "N/A" 

    try:
        location = "N/A"
        for comment in record['comments']:
            if comment.get('commentType') == 'SUBCELLULAR LOCATION':
                location = comment['subcellularLocations'][0]['location']['value']
                break
    except (KeyError, IndexError, TypeError):
        location = "N/A"


    try:
        taxon_id = record['organism']['taxonId']
    except KeyError:
        taxon_id = "N/A"

    return {
        'UniProtID': up_id, 
        'EntryName': entry_name,
        'ProteinName': description,
        'GeneName': gene_name,
        'Length': length,
        'Organism': organism, 
        'Function': function,
        'CellularLocation': location,
        'TaxonID': taxon_id 
        
    }


res_summary = []
for record in results:
    res_summary.append(get_record_dict(record))
    # limit to 25 records for display purposes (and speed!)
    if len(res_summary)>= 25:
        break

summary_df = pd.DataFrame(res_summary)

summary_df

,UniProtID,EntryName,ProteinName,GeneName,Length,Organism,Function,CellularLocation,TaxonID
0,Q0VDF9,HSP7E_HUMAN,Heat shock 70 kDa protein 14,HSPA14,509,Homo sapiens,Component of the ribosome-associated complex (...,"Cytoplasm, cytosol",9606
1,P34931,HS71L_HUMAN,Heat shock 70 kDa protein 1-like,HSPA1L,641,Homo sapiens,Molecular chaperone implicated in a wide varie...,N/A,9606
2,Q9Y4L1,HYOU1_HUMAN,Hypoxia up-regulated protein 1,HYOU1,999,Homo sapiens,Has a pivotal role in cytoprotective cellular ...,Endoplasmic reticulum lumen,9606
3,P11021,BIP_HUMAN,Endoplasmic reticulum chaperone BiP,HSPA5,654,Homo sapiens,Endoplasmic reticulum chaperone that plays a k...,Endoplasmic reticulum lumen,9606
4,P54652,HSP72_HUMAN,Heat shock-related 70 kDa protein 2,HSPA2,639,Homo sapiens,Molecular chaperone implicated in a wide varie...,"Cytoplasm, cytoskeleton, spindle",9606
5,P0DMV9,HS71B_HUMAN,Heat shock 70 kDa protein 1B,HSPA1B,641,Homo sapiens,Molecular chaperone implicated in a wide varie...,Cytoplasm,9606
6,P11142,HSP7C_HUMAN,Heat shock cognate 71 kDa protein,HSPA8,646,Homo sapiens,Molecular chaperone implicated in a wide varie...,Cytoplasm,9606
7,P17066,HSP76_HUMAN,Heat shock 70 kDa protein 6,HSPA6,643,Homo sapiens,Molecular chaperone implicated in a wide varie...,N/A,9606
8,P38646,HSPA9_HUMAN,"Stress-70 protein, mitochondrial",HSPA9,679,Homo sapiens,Mitochondrial chaperone that plays a key role ...,Mitochondrion,9606
9,P48723,HSP13_HUMAN,Heat shock 70 kDa protein 13,HSPA13,471,Homo sapiens,Has peptide-independent ATPase activity,Microsome,9606


In [16]:
url = "https://rest.uniprot.org/uniprotkb/search"

In [17]:
params = { 
    "query": "accession:p11021 OR accession:p0A6Y8 OR accession:p38647", 
    "format": "fasta"
} 
response = requests.get(url, params=params) 


In [18]:
with open("HSP70.fasta", "w") as fh:
    fh.writelines(response.text)

In [21]:
%pip install rcsb-api

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 19.7 MB/s  0:00:00

   ---------------------------------------- 0/8 [tqdm]
   ---------------------------------------- 0/8 [tqdm]
   ---------------------------------------- 0/8 [tqdm]
   ---------------------------------------- 0/8 [tqdm]
   ---------------------------------------- 0/8 [tqdm]
   ----- ---------------------------------- 1/8 [rustworkx]
   ---------- ----------------------------- 2/8 [h11]
   ---------- ----------------------------- 2/8 [h11]
   ---------- ----------------------------- 2/8 [h11]
   --------------- ------------------------ 3/8 [graphql-core]
   --------------- ------------------------ 3/8 [graphql-core]
   --------------- ------------------------ 3/8 [graphql-core]
   --------------- ------------------------ 3/8 [graphql-core]
   --------------- ------------------------ 3/8 [graphql-core]
   --------------- ------------------------ 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from rcsbapi.search import StructSimilarityQuery

reference_id = "4B9Q"   

q_struct = StructSimilarityQuery(
    structure_search_type="entry_id",
    entry_id=reference_id,
    chain_id="A",
    similarity_type="global",
    target_search_space="polymer_entity_instance"
)
list(q_struct()) 

['4B9Q',
 '7KO2',
 '5NRO',
 '4JNE',
 '7KRT',
 '4JN4',
 '5E84',
 '7N1R',
 '6ASY',
 '2KHO',
 '7KZI',
 '7SQC',
 '7N6G',
 '5TKY',
 '9BLS',
 '9BLT',
 '7KRW',
 '3D2E',
 '3D2F',
 '2QXL',
 '3C7N',
 '1YUW',
 '7KRV',
 '7KRU',
 '4J8F',
 '7KZU',
 '7PLK']

In [30]:
similar_ids = []
for result in q_struct():
    # results come back as "PDBID.chain" e.g. "4B9Q.A" — extract just the PDB ID
    pdb_id = result.split(".")[0]
    if pdb_id not in similar_ids and pdb_id != reference_id:
        similar_ids.append(pdb_id)
    if len(similar_ids) >= 25:
        break


In [31]:
print(f"Found {len(similar_ids)} structurally similar HSP70 entries")
print(similar_ids)

Found 25 structurally similar HSP70 entries
['7KO2', '5NRO', '4JNE', '7KRT', '4JN4', '5E84', '7N1R', '6ASY', '2KHO', '7KZI', '7SQC', '7N6G', '5TKY', '9BLS', '9BLT', '7KRW', '3D2E', '3D2F', '2QXL', '3C7N', '1YUW', '7KRV', '7KRU', '4J8F', '7KZU']


In [38]:
def get_pdb_record_dict(pdb_id):
    """
    Fetch and extract structural data from a PDB entry via the RCSB Data API.
    Hits three endpoints: entry, polymer_entity, and polymer_entity_instance.

    Args:
        pdb_id (str): A PDB entry ID (e.g. '4B9Q').

    Returns:
        dict: A dictionary containing the extracted structural fields.
    """

    base = "https://data.rcsb.org/rest/v1/core"

    # ── Endpoint 1: Entry-level metadata ──────────────────────────────────
    entry_resp = requests.get(f"{base}/entry/{pdb_id}")
    entry = entry_resp.json() if entry_resp.status_code == 200 else {}

    try:
        title = entry['struct']['title']
    except KeyError:
        title = "N/A"

    try:
        exp_method = entry['exptl'][0]['method']
    except (KeyError, IndexError):
        exp_method = "N/A"

    try:
        resolution = entry['rcsb_entry_info']['resolution_combined'][0]
    except (KeyError, IndexError):
        resolution = "N/A"

    try:
        deposition_date = entry['rcsb_accession_info']['deposit_date']
    except KeyError:
        deposition_date = "N/A"

    # ── Endpoint 2: Polymer entity (chain 1) ──────────────────────────────
    entity_resp = requests.get(f"{base}/polymer_entity/{pdb_id}/1")
    entity = entity_resp.json() if entity_resp.status_code == 200 else {}

    try:
        organism = entity['rcsb_entity_source_organism'][0]['scientific_name']
    except (KeyError, IndexError):
        organism = "N/A"

    try:
        taxonomy_id = entity['rcsb_entity_source_organism'][0]['ncbi_taxonomy_id']
    except (KeyError, IndexError):
        taxonomy_id = "N/A"

    try:
        seq_length = entity['entity_poly']['rcsb_sample_sequence_length']
    except KeyError:
        seq_length = "N/A"

    try:
        sequence = entity['entity_poly']['pdbx_seq_one_letter_code_can']
    except KeyError:
        sequence = "N/A"

    # ── Endpoint 3: Polymer entity instance (chain A) ─────────────────────
    instance_resp = requests.get(f"{base}/polymer_entity_instance/{pdb_id}/A")
    instance = instance_resp.json() if instance_resp.status_code == 200 else {}

    return {
        'PDB_ID':          pdb_id,
        'Title':           title,
        'Organism':        organism,
        'Taxonomy_ID':     taxonomy_id,
        'Exp_Method':      exp_method,
        'Resolution_A':    resolution,
        'Seq_Length':      seq_length,
        'Sequence':        sequence,
    }

In [39]:
pdb_summary = []
for pdb_id in similar_ids:
    pdb_summary.append(get_pdb_record_dict(pdb_id))
    print(f"Fetched: {pdb_id}")  # progress tracker

pdb_df = pd.DataFrame(pdb_summary)
pdb_df

Fetched: 7KO2
Fetched: 5NRO
Fetched: 4JNE
Fetched: 7KRT
Fetched: 4JN4
Fetched: 5E84
Fetched: 7N1R
Fetched: 6ASY
Fetched: 2KHO
Fetched: 7KZI
Fetched: 7SQC
Fetched: 7N6G
Fetched: 5TKY
Fetched: 9BLS
Fetched: 9BLT
Fetched: 7KRW
Fetched: 3D2E
Fetched: 3D2F
Fetched: 2QXL
Fetched: 3C7N
Fetched: 1YUW
Fetched: 7KRV
Fetched: 7KRU
Fetched: 4J8F
Fetched: 7KZU


,PDB_ID,Title,Organism,Taxonomy_ID,Exp_Method,Resolution_A,Seq_Length,Sequence
0,7KO2,Restraining state of near full-length Hsp70 DnaK,Escherichia coli (strain K12),83333,X-RAY DIFFRACTION,2.64,609,MGKIIGIDLGTTNSCVAIMDGTTPRVLENAEGDRTTPSIIAYTQDG...
1,5NRO,Structure of full-length DnaK with bound J-domain,Escherichia coli,562,X-RAY DIFFRACTION,3.25,611,MGKIIGIDLGTTNSCVAIMDGTTPRVLENAEGDRTTPSIIAYTQDG...
2,4JNE,Allosteric opening of the polypeptide-binding ...,Escherichia coli,83333,X-RAY DIFFRACTION,1.96,608,SGKIIGIDLGTTNSCVAIMDGTTPRVLENAEGDRTTPSIIAYTQDG...
3,7KRT,Restraining state of a truncated Hsp70 DnaK,Escherichia coli (strain K12),83333,X-RAY DIFFRACTION,2.79,600,MGKIIGIDLGTTNSCVAIMDGTTPRVLENAEGDRTTPSIIAYTQDG...
4,4JN4,Allosteric opening of the polypeptide-binding ...,Escherichia coli,83333,X-RAY DIFFRACTION,2.3,608,SGKIIGIDLGTTNSCVAIMDGTTPRVLENAEGDRTTPSIIAYTQDG...
5,5E84,ATP-bound state of BiP,Homo sapiens,9606,X-RAY DIFFRACTION,2.99,606,SEDVGTVVGIDLGTTYSCVGVFKNGRVEIIANDQGNRITPSYVAFT...
6,7N1R,A novel and unique ATP hydrolysis to AMP by a ...,Homo sapiens,9606,X-RAY DIFFRACTION,2.03,606,SEDVGTVVGIDLGTTYSCVGVFKNGRVEIIANDQGNRITPSYVAFT...
7,6ASY,BiP-ATP2,Homo sapiens,9606,X-RAY DIFFRACTION,1.85,606,SEDVGTVVGIDLGTTYSCVGVFKNGRVEIIANDQGNRITPSYVAFT...
8,2KHO,NMR-RDC / XRAY structure of E. coli HSP70 (DNA...,Escherichia coli,83333,SOLUTION NMR,N/A,605,MGKIIGIDLGTTNSCVAIMDGTTPRVLENAEGDRTTPSIIAYTQDG...
9,7KZI,Intermediate state (QQQ) of near full-length D...,Escherichia coli (strain K12),83333,X-RAY DIFFRACTION,2.82,628,MGKIIGIDLGTTNSCVAIMDGTTPRVLENAEGDRTTPSIIAYTQDG...
